# Projeto Data Warehouse
#### Esse projeto tem o objetivo de realizar um **Data Warehouse**, criando as tabelas no **banco de dados postgres** e inserindo as informações tratadas da base *dimensão* e *fato*. As informações seram apresentada através da biblioteca **Gradio** via gráficos web.

#### PASSO 1: Preparando Ambiente:

In [ ]:
%pip install -q pandas gradio
%pip install -q openpyxl
%pip install -q sqlalchemy psycopg2-binary sqlalchemy-utils
%pip install -q matplotlib
%pip install -q plotly

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import gradio as gr
import os
from sqlalchemy import create_engine
from sqlalchemy_utils import database_exists,create_database
import psycopg2
import warnings

warnings.filterwarnings('ignore')

c:\Users\Daniel Gama\Documents\Computação\Github\analise_dados\Python\ETL-ProejtoDaniel\venv-projetoDaniel\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def details_df2(df, opc):
    try:
        if opc == 'tamanho':
            display(f'Tamanho (linhas, colunas): {df.shape}')
        elif opc == 'linha':
            display(f'\nQuantidade Linhas: {df.shape[0]}')
        elif opc == 'coluna':
            nomes_colunas = df.columns.tolist()
            display(f'Quantidade de Colunas: {df.shape[1]}')
            display(f'Nomes das Colunas: {nomes_colunas}')
        elif opc == 'info':
            # Apenas chame df.info() diretamente
            print("\nInfo:")
            df.info()
        elif opc == 'head':
            # df.head() retorna um DataFrame, então imprima-o diretamente
            print("\nHead:")
            display(df.head())
        elif opc == 'isnull':
            # df.isnull().sum() retorna uma Series, então imprima-a diretamente
            print('\nIsnull:')
            display(df.isnull().sum())
        elif opc == 'todos':
            # Para 'todos', é melhor quebrar para não misturar print com retorno None
            print(f"Tamanho (linhas, colunas):{df.shape}")
            print("\nInfo:")
            df.info()
            print("\nHead:")
            print(df.head())
            print("\nIsnull:")
            print(df.isnull().sum())
        elif opc == 'config':
            print('Informações sobre configuração detals_df:\nEstá função pode mostrar detalhes no campo opc:\ntamanho: Tamanho do DF.\nlinha: Quantidade de linhas no DF.' \
            '\ncoluna: Quantidade de colunas e nomes das colunas no DF.\ninfo: Puxa informações do DF.\nhead: Puxa o head (cabeçalho) de 5 linhas do DF.\nisnull: Informa campos vazios/nulos agrupado por colunas.' \
            '\ntodos: Puxa todas as informações. Obs: Menos linha e coluna.')
        else:
            print('Erro: Informe "opc" entre tamanho, linha, coluna, info, head, isnull ou todos')
    except Exception as e:
        print(f'Erro inesperado na função detals_df: {e}')
        

def describe_df(df,opc):
    try:
        if opc == 'normal':
            display(df.describe())
        elif opc == 'object':
            display(df.describe(include='object'))
        else:
            print('Erro: Informe campo "opc" entre normal e object!')
    except Exception as e:
            print(f'Erro: Inesperado na função describe_df: {e}')

def type_columns_df(df,opc):
    try:
        if opc == 'quantitativa':
            variaveis_quantitativas = df.select_dtypes(include=['number']).columns.tolist()
            print(f"Variáveis Quantitativas: {variaveis_quantitativas}")
            return variaveis_quantitativas
        elif opc == 'qualitativa':
            variaveis_qualitativas = df.select_dtypes(include=["category","object"]).columns.tolist()
            print(f"Variáveis Qualitativa: {variaveis_qualitativas}")
            return variaveis_qualitativas
        elif opc == 'todos':
            variaveis_quantitativas = df.select_dtypes(include=['number']).columns.tolist()
            print(f"Variáveis Quantitativas: {variaveis_quantitativas}")
            variaveis_qualitativas = df.select_dtypes(include=["category","object"]).columns.tolist()
            print(f"Variáveis Qualitativa: {variaveis_qualitativas}")
            # Retorna ambas as listas
            return variaveis_quantitativas, variaveis_qualitativas
        else:
            print('Erro: Escolha no campo "opc" entre "quantitativa", "qualitativa" ou "todos"')
            return None # Retorna None explicitamente para indicar falha ou opção inválida
    except Exception as e:
            print(f'Erro: Inesperado na função tipy_columns_df: {e}')
            return None 

def renames_columns_df(df,dicionario,boolean):
    try:
        if boolean:
            df.rename(columns=dicionario,inplace=boolean)
            print('Coluna(s) informada(s) no dicionário foram renomeadas com sucesso!')
            details_df(df,'head')
        elif boolean == False:
            df_final = df.rename(columns=dicionario)
            print('Coluna(s) informada(s) no dicionário foram renomeadas com sucesso!')
            details_df(df_final,'head')
            return df_final
        else:
            print('Erro: Informe o tipo de Inplace True ou False, caso True passe variável para receber informação.')
    except Exception as e:
        print(f'Erro: Inesperado na função renames_columns_df: {e}')
        return None 

def renames_fields_df(df,coluna,dicionario):
    try:
        df[coluna] =  df[coluna].replace(dicionario)
        print(df[coluna].value_counts(dropna=False))
        return df
    except Exception as e:
        print(f'Erro: Inesperado na função renames_fields_df: {e}')
        return None

def value_counts_df(df,opc,coluna):
    try:    
        if opc == 'normal':
            print(f'\n{df[coluna].value_counts(dropna=False)}')
        elif opc == 'proporcao':
            prop = df[coluna].value_counts(normalize=True).reset_index().rename(columns={"proportion":"Proporção"}).sort_values(by="Proporção", ascending=False)
            print(f'\n{prop}')
        elif opc == 'todos':
            print(f'\n{df[coluna].value_counts(dropna=False)}')
            print(f'\n{df[coluna].value_counts(normalize=True).reset_index().rename(columns={"proportion":"Proporção"}).sort_values(by="Proporção", ascending=False)}')
        elif opc == 'config':
            print('\nInformações sobre configuração value_counts_df:\nnormal: Retorna as categorias e suas quantidades de uma determina coluna.\nproporcao:  Retorna as categorias e suas proporção de uma determina coluna.\n' \
            'todos: Retorna as categorias, suas quantidades e proporção.')
        else:
            print('\nErro: Informe a "opc" entre normal, proporcao, todos e config para mais detalhes.')
    except Exception as e:
            print(f'Erro: Inesperado na função value_counts_df: {e}')

def str_upper_df(df,column):
    try:
        df[column] = df[column].str.upper()
        print(f'\nAs categorias contidas na coluna {column} foram convertidas para maiúsculas!')
    except Exception as e:
        print(f'Erro: Inesperado na função str_upper_df: {e}')

def drop_column_df(df,columns,boolean):
    try:
        if boolean:
            df.drop(columns=columns, inplace=boolean)
            print(f'Os campos {columns} foram excluídos com sucesso!')
            details_df(df,'head')
        elif boolean == False:
            df_final = df.drop(columns=columns, inplace=boolean)
            print(f'Os campos {columns} foram excluídos com sucesso!')
            details_df(df_final,'head')
            return df_final
        else:
            print('Erro: Informe o inplace True ou False, caso True passe variável para receber informação.')
    except Exception as e:
        print(f'Erro: Inesperado na função drop_column_df: {e}')


def convert_type_df(df,opc,coluna,tipo=None):
        try:
            if opc == 'outro':
                df[coluna] = df[coluna].astype(tipo)
                print(f'A coluna {coluna}, foi alterada para {tipo} com sucesso!')
                return df[coluna]
            elif opc == 'datetime':
                df[coluna]= pd.to_datetime(df[coluna], format='%d/%m/%Y')
                print(f'A coluna {coluna}, foi alterada para datetime com sucesso!')
                return df[coluna]
            else:
                print('Erro: Por favor escolha "opc" entre outros para definir tipos Ex: int ou datetime para conversão em datas.')
        except Exception as e:
            print(f'Erro: Inesperado na função convert_type_df: {e}')
            if opc == 'datetime':
                return df

#======================================Melhorias Gemini===================================================
def details_df(df, opc):
    """
    Exibe detalhes de um DataFrame com base nas opções fornecidas.

    Args:
        df (pandas.DataFrame): O DataFrame a ser analisado.
        opc (str ou list): Uma string ou lista de strings especificando quais detalhes exibir.
                           Opções válidas: 'tamanho', 'linha', 'coluna', 'info', 'head', 'isnull', 'config'.
    """
    if isinstance(opc, str):
        opc = [opc]  # Converte a string em uma lista para uniformizar o tratamento

    for item in opc:
        try:
            if item == 'tamanho':
                print("\n--- Tamanho do DataFrame ---")
                display(f'Tamanho (linhas, colunas): {df.shape}')
            elif item == 'linha':
                print("\n--- Quantidade de Linhas ---")
                display(f'Quantidade Linhas: {df.shape[0]}')
            elif item == 'coluna':
                print("\n--- Informações de Colunas ---")
                nomes_colunas = df.columns.tolist()
                display(f'Quantidade de Colunas: {df.shape[1]}')
                display(f'Nomes das Colunas: {nomes_colunas}')
            elif item == 'info':
                print("\n--- Informações Gerais (df.info()) ---")
                df.info()
            elif item == 'head':
                print("\n--- Primeiras 5 Linhas (df.head()) ---")
                display(df.head())
            elif item == 'isnull':
                print("\n--- Valores Nulos por Coluna (df.isnull().sum()) ---")
                display(df.isnull().sum())
            elif item == 'todos':
                display(f"Tamanho (linhas, colunas): {df.shape}")
                print("\n--- Informações de Colunas ---")
                nomes_colunas = df.columns.tolist()
                display(f'Quantidade de Colunas: {df.shape[1]}')
                display(f'Nomes das Colunas: {nomes_colunas}')
                print("\n--- Informações Gerais (df.info()) ---")
                df.info()
                print("\n--- Primeiras 5 Linhas (df.head()) ---")
                display(df.head())
                print("\n--- Valores Nulos por Coluna (df.isnull().sum()) ---")
                display(df.isnull().sum())
            elif item == 'config':
                print('\n--- Configurações de Uso da Função details_df ---')
                print('Esta função pode mostrar detalhes no campo "opc".')
                print('Você pode passar uma string ou uma lista de strings com as seguintes opções:')
                print('  - "tamanho": Tamanho (linhas, colunas) do DF.')
                print('  - "linha": Quantidade de linhas no DF.')
                print('  - "coluna": Quantidade de colunas e nomes das colunas no DF.')
                print('  - "info": Informações detalhadas do DF (df.info()).')
                print('  - "head": As 5 primeiras linhas do DF (df.head()).')
                print('  - "isnull": Contagem de valores nulos por coluna (df.isnull().sum()).')
                print('  - "config": Exibe esta mensagem de configurações.')
            else:
                print(f'\n--- Opção Inválida: "{item}" ---')
                print('Opções válidas: "tamanho", "linha", "coluna", "info", "head", "isnull", "config".')
        except Exception as e:
            print(f'\n--- Erro ao processar "{item}": {e} ---')

def merge_df(df1, df2, merge_params, columns_to_drop=None):
    """
    Realiza um merge entre dois DataFrames, permite a remoção de colunas
    e exibe o cabeçalho do DataFrame resultante.

    Args:
        df1 (pd.DataFrame): O primeiro DataFrame (left).
        df2 (pd.DataFrame): O segundo DataFrame (right).
        merge_params (list): Uma lista contendo [left_on, right_on, how].
                             - left_on (str ou list): Coluna(s) para unir em df1.
                             - right_on (str ou list): Coluna(s) para unir em df2.
                             - how (str): Tipo de merge ('left', 'right', 'inner', 'outer').
        columns_to_drop (list, optional): Uma lista de nomes de colunas a serem removidas
                                           do DataFrame resultante. Padrão para None.

    Returns:
        pd.DataFrame: O DataFrame resultante da operação de merge.
    """
    if len(merge_params) != 3:
        raise ValueError("merge_params deve conter [left_on, right_on, how].")

    left_on, right_on, how = merge_params

    print(f"Realizando merge '{how}' entre DataFrames...")
    print(f"Chave(s) no primeiro DF (left_on): {left_on}")
    print(f"Chave(s) no segundo DF (right_on): {right_on}")

    try:
        df_merged = pd.merge(df1, df2, left_on=left_on, right_on=right_on, how=how)
        print("\nMerge concluído com sucesso!")
        print(f"Dimensões do DataFrame resultante: {df_merged.shape}")

        if columns_to_drop:
            print(f"\nRemovendo as seguintes colunas: {columns_to_drop}")
            # Verifica se as colunas a serem dropadas existem no DataFrame
            existing_columns = [col for col in columns_to_drop if col in df_merged.columns]
            non_existing_columns = [col for col in columns_to_drop if col not in df_merged.columns]

            if non_existing_columns:
                print(f"Atenção: As colunas {non_existing_columns} não foram encontradas no DataFrame e não serão removidas.")

            if existing_columns:
                df_merged = df_merged.drop(columns=existing_columns)
                print("Colunas removidas com sucesso!")
                print(f"Novas dimensões do DataFrame: {df_merged.shape}")
            else:
                print("Nenhuma das colunas especificadas para remoção foi encontrada no DataFrame.")

        print("\n--- Head do DataFrame Resultante ---")
        display(df_merged.head())

        return df_merged

    except Exception as e:
        print(f"Erro ao realizar a operação de merge: {e}")
        return None

#### PASSO 2: Extração:

In [4]:
canal_df = pd.read_excel('./Dados/Dimensão/Canal.xlsx')
categoria_df = pd.read_excel('./Dados/Dimensão/Categoria.xlsx')
cidade_df = pd.read_excel('./Dados/Dimensão/Cidade.xlsx')
marca_df = pd.read_excel('./Dados/Dimensão/Marca.xlsx')
produto_df = pd.read_excel('./Dados/Dimensão/Produto.xlsx')
subcategoria_df = pd.read_excel('./Dados/Dimensão/Subcategoria.xlsx')
clientes_df = pd.read_csv('./Dados/Dimensão/Clientes.txt',sep='\t', encoding='utf-16-le')

In [101]:
details_df(clientes_df,['head','info','isnull'])


--- Primeiras 5 Linhas (df.head()) ---


,ID_Cliente,Primeiro Nome,Sobrenome,Data Nascimento,Estado Civil,Gênero,Educação,ID_Cidade
0,5000,Ruben,Jimenez,'12-11-1966',Solteiro(a),Masculino,Mestrado,19
1,5001,Arthur,Sai,'5-14-1966',Casado(a),Masculino,Mestrado,23
2,5002,Mayra,Sai,'2-24-1966',Casado(a),Feminino,Mestrado,4
3,5003,Grant,Ferrier,'7-27-1966',Casado(a),Masculino,Mestrado,14
4,5004,Angela,Cook,'11-6-1965',Solteiro(a),Feminino,Ensino Fundamental,30



--- Informações Gerais (df.info()) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18484 entries, 0 to 18483
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   ID_Cliente       18484 non-null  int64 
 1   Primeiro Nome    18484 non-null  object
 2   Sobrenome        18484 non-null  object
 3   Data Nascimento  18484 non-null  object
 4   Estado Civil     18484 non-null  object
 5   Gênero           18484 non-null  object
 6   Educação         18484 non-null  object
 7   ID_Cidade        18484 non-null  int64 
dtypes: int64(2), object(6)
memory usage: 1.1+ MB

--- Valores Nulos por Coluna (df.isnull().sum()) ---


ID_Cliente         0
Primeiro Nome      0
Sobrenome          0
Data Nascimento    0
Estado Civil       0
Gênero             0
Educação           0
ID_Cidade          0
dtype: int64

In [6]:
details_df(canal_df,['head','info','isnull'])


--- Primeiras 5 Linhas (df.head()) ---


,ID_Canal,Descricao_Canal
0,1,Internet
1,2,Loja Física



--- Informações Gerais (df.info()) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   ID_Canal         2 non-null      int64 
 1   Descricao_Canal  2 non-null      object
dtypes: int64(1), object(1)
memory usage: 164.0+ bytes

--- Valores Nulos por Coluna (df.isnull().sum()) ---


ID_Canal           0
Descricao_Canal    0
dtype: int64

In [7]:
details_df(categoria_df,['head','info','isnull'])


--- Primeiras 5 Linhas (df.head()) ---


,ID_Categoria,Categoria
0,1,Celulares
1,2,Televisores
2,3,Informática
3,4,Games
4,5,Eletrodomésticos



--- Informações Gerais (df.info()) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   ID_Categoria  7 non-null      int64 
 1   Categoria     7 non-null      object
dtypes: int64(1), object(1)
memory usage: 244.0+ bytes

--- Valores Nulos por Coluna (df.isnull().sum()) ---


ID_Categoria    0
Categoria       0
dtype: int64

In [8]:
details_df(cidade_df,['head','info','isnull'])


--- Primeiras 5 Linhas (df.head()) ---


,ID_Cidade,Cidade,UF
0,1,São Paulo,SP
1,2,Guarulhos,SP
2,3,Curitiba,PR
3,4,Joinville,SC
4,5,Florianópolis,SC



--- Informações Gerais (df.info()) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   ID_Cidade  31 non-null     int64 
 1   Cidade     31 non-null     object
 2   UF         31 non-null     object
dtypes: int64(1), object(2)
memory usage: 876.0+ bytes

--- Valores Nulos por Coluna (df.isnull().sum()) ---


ID_Cidade    0
Cidade       0
UF           0
dtype: int64

In [9]:
details_df(marca_df,['head','info','isnull'])


--- Primeiras 5 Linhas (df.head()) ---


,ID_Marca,Marca
0,1,Apple
1,2,Samsung
2,3,Motorola
3,4,Xiaomi
4,5,LG



--- Informações Gerais (df.info()) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   ID_Marca  21 non-null     int64 
 1   Marca     21 non-null     object
dtypes: int64(1), object(1)
memory usage: 468.0+ bytes

--- Valores Nulos por Coluna (df.isnull().sum()) ---


ID_Marca    0
Marca       0
dtype: int64

In [10]:
details_df(produto_df,['head','info','isnull'])


--- Primeiras 5 Linhas (df.head()) ---


,ID_Subcategoria,ID_Produto,Descricao_Produto,ID_Marca,Preco_Unitario,Tributos,Custo
0,10,1010,iPhone 11,1,2899,347.88,550.81
1,10,1011,iPhone 12,1,3499,419.88,699.80
2,10,1012,iPhone 13,1,4099,614.85,655.84
3,10,1013,iPhone 13 Pro,1,4299,558.87,773.82
4,10,1014,iPhone 13 Pro Max,1,4399,615.86,395.91



--- Informações Gerais (df.info()) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 213 entries, 0 to 212
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   ID_Subcategoria    213 non-null    int64  
 1   ID_Produto         213 non-null    int64  
 2   Descricao_Produto  213 non-null    object 
 3   ID_Marca           213 non-null    int64  
 4   Preco_Unitario     213 non-null    int64  
 5   Tributos           213 non-null    float64
 6   Custo              213 non-null    float64
dtypes: float64(2), int64(4), object(1)
memory usage: 11.8+ KB

--- Valores Nulos por Coluna (df.isnull().sum()) ---


ID_Subcategoria      0
ID_Produto           0
Descricao_Produto    0
ID_Marca             0
Preco_Unitario       0
Tributos             0
Custo                0
dtype: int64

In [11]:
details_df(subcategoria_df,['head','info','isnull'])


--- Primeiras 5 Linhas (df.head()) ---


,ID_Categoria,ID_Subcategoria,Subcategoria
0,1,10,IOS
1,1,11,Android
2,2,12,LED
3,2,13,QLED
4,2,14,OLED



--- Informações Gerais (df.info()) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   ID_Categoria     22 non-null     int64 
 1   ID_Subcategoria  22 non-null     int64 
 2   Subcategoria     22 non-null     object
dtypes: int64(2), object(1)
memory usage: 660.0+ bytes

--- Valores Nulos por Coluna (df.isnull().sum()) ---


ID_Categoria       0
ID_Subcategoria    0
Subcategoria       0
dtype: int64

In [32]:
# PEGA AS TABELAS A PARTIR DA 5 LINHA DE CADA TABELA,  QUE ESTÃO DENTRO DO CAMINHO FATO e JUNTA EM UMA ÚNICA TABELA
    #Caminho para a pasta com os arquivos
pasta = r'.\Dados\Fato'

#lista com todos os arquivos dentro da pasta
arquivos = [os.path.join(pasta,arquivo) for arquivo in os.listdir(pasta)]

#tabela começa vazia
tabela_final = pd.DataFrame()

#loop para iterar sobre todos os arquivos
#e ir juntando as informações na "tabale_final"
for arquivo in arquivos:
    #ler arquivo excel pulando 4 linhas
    df = pd.read_excel(arquivo, skiprows=4)
#junta informações do arquivo na tabela
tabela_final= pd.concat([tabela_final,df])

#exporta tabela para uma planilha excel
tabela_final.to_excel("./Dados/Fato/fato_final.xlsx", index=False)
print(tabela_final)


      Data_Venda Data_Entrega  ID_Canal  ID_Cliente  ID_Pedido  ID_Produto  \
0     2018-01-01   2018-01-22         2        5010      66417        1038   
1     2018-01-01   2018-02-05         1        5426      66418        1177   
2     2018-01-01   2018-01-02         2        7499      66419        1198   
3     2018-01-01   2018-02-05         2        8080      66420        1049   
4     2018-01-01   2018-01-28         2        9370      66421        1102   
...          ...          ...       ...         ...        ...         ...   
33540 2021-12-29   2022-01-31         1       21408      99938        1130   
33541 2021-12-29   2022-01-17         2       21692      99939        1011   
33542 2021-12-29   2022-01-18         2       21835      99940        1032   
33543 2021-12-29   2022-01-14         1       22737      99941        1040   
33544 2021-12-29   2021-12-30         1       23427      99942        1047   

       Qtde  Valor Total  
0        15     26445.30  
1        

In [33]:
fato_df = pd.read_excel('./Dados/Fato/fato_final.xlsx')

In [13]:
details_df(tabela_final,['head','info','isnull'])


--- Primeiras 5 Linhas (df.head()) ---


,Data_Venda,Data_Entrega,ID_Canal,ID_Cliente,ID_Pedido,ID_Produto,Qtde,Valor Total
0,2018-01-01,2018-01-22,2,5010,66417,1038,15,26445.30
1,2018-01-01,2018-02-05,1,5426,66418,1177,7,34825.91
2,2018-01-01,2018-01-02,2,7499,66419,1198,12,11946.24
3,2018-01-01,2018-02-05,2,8080,66420,1049,25,79176.00
4,2018-01-01,2018-01-28,2,9370,66421,1102,16,95957.60



--- Informações Gerais (df.info()) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33545 entries, 0 to 33544
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Data_Venda    33545 non-null  datetime64[ns]
 1   Data_Entrega  33545 non-null  datetime64[ns]
 2   ID_Canal      33545 non-null  int64         
 3   ID_Cliente    33545 non-null  int64         
 4   ID_Pedido     33545 non-null  int64         
 5   ID_Produto    33545 non-null  int64         
 6   Qtde          33545 non-null  int64         
 7   Valor Total   33545 non-null  float64       
dtypes: datetime64[ns](2), float64(1), int64(5)
memory usage: 2.0 MB

--- Valores Nulos por Coluna (df.isnull().sum()) ---


Data_Venda      0
Data_Entrega    0
ID_Canal        0
ID_Cliente      0
ID_Pedido       0
ID_Produto      0
Qtde            0
Valor Total     0
dtype: int64

### PASSO 3: Tratamento.

In [102]:
renames_columns_df(clientes_df,{'ID_Cliente':'id','Primeiro Nome':'nome','Sobrenome':'sobrenome','Data Nascimento':'data_nascimento','Estado Civil':'estado_civil','Gênero':'genero','Educação':'educacao','ID_Cidade':'id_cidade'},True)

Coluna(s) informada(s) no dicionário foram renomeadas com sucesso!

--- Primeiras 5 Linhas (df.head()) ---


,id,nome,sobrenome,data_nascimento,estado_civil,genero,educacao,id_cidade
0,5000,Ruben,Jimenez,'12-11-1966',Solteiro(a),Masculino,Mestrado,19
1,5001,Arthur,Sai,'5-14-1966',Casado(a),Masculino,Mestrado,23
2,5002,Mayra,Sai,'2-24-1966',Casado(a),Feminino,Mestrado,4
3,5003,Grant,Ferrier,'7-27-1966',Casado(a),Masculino,Mestrado,14
4,5004,Angela,Cook,'11-6-1965',Solteiro(a),Feminino,Ensino Fundamental,30


In [103]:
print("Removendo aspas simples da coluna 'data_nascimento'...")
# Usamos .astype(str) para garantir que a coluna seja string antes de aplicar .str.replace
clientes_df['data_nascimento'] = clientes_df['data_nascimento'].astype(str).str.replace("'", "")
print("Aspas removidas.")

# 3. Converter a coluna 'data_nascimento' para o tipo datetime
print("Convertendo 'data_nascimento' para o tipo DATE...")
try:
    # Tentamos novamente com o formato Mês-Dia-Ano, agora sem aspas
    clientes_df['data_nascimento'] = pd.to_datetime(clientes_df['data_nascimento'], format='%m-%d-%Y')
    print("Conversão de 'data_nascimento' realizada com sucesso.")
except Exception as e:
    print(f"ERRO FINAL: Não foi possível converter 'data_nascimento' mesmo após remover aspas: {e}")

Removendo aspas simples da coluna 'data_nascimento'...
Aspas removidas.
Convertendo 'data_nascimento' para o tipo DATE...
Conversão de 'data_nascimento' realizada com sucesso.


In [104]:
details_df(clientes_df,['head'])


--- Primeiras 5 Linhas (df.head()) ---


,id,nome,sobrenome,data_nascimento,estado_civil,genero,educacao,id_cidade
0,5000,Ruben,Jimenez,1966-12-11,Solteiro(a),Masculino,Mestrado,19
1,5001,Arthur,Sai,1966-05-14,Casado(a),Masculino,Mestrado,23
2,5002,Mayra,Sai,1966-02-24,Casado(a),Feminino,Mestrado,4
3,5003,Grant,Ferrier,1966-07-27,Casado(a),Masculino,Mestrado,14
4,5004,Angela,Cook,1965-11-06,Solteiro(a),Feminino,Ensino Fundamental,30


In [14]:
renames_columns_df(canal_df,{'ID_Canal':'id','Descricao_Canal':'descricao_canal'},True)

Coluna(s) informada(s) no dicionário foram renomeadas com sucesso!

--- Primeiras 5 Linhas (df.head()) ---


,id,descricao_canal
0,1,Internet
1,2,Loja Física


In [15]:
renames_columns_df(categoria_df,{'ID_Categoria':'id','Categoria':'categoria'},True)

Coluna(s) informada(s) no dicionário foram renomeadas com sucesso!

--- Primeiras 5 Linhas (df.head()) ---


,id,categoria
0,1,Celulares
1,2,Televisores
2,3,Informática
3,4,Games
4,5,Eletrodomésticos


In [16]:
renames_columns_df(cidade_df,{'ID_Cidade':'id','Cidade':'cidade','UF':'uf'},True)

Coluna(s) informada(s) no dicionário foram renomeadas com sucesso!

--- Primeiras 5 Linhas (df.head()) ---


,id,cidade,uf
0,1,São Paulo,SP
1,2,Guarulhos,SP
2,3,Curitiba,PR
3,4,Joinville,SC
4,5,Florianópolis,SC


In [17]:
renames_columns_df(marca_df,{'ID_Marca':'id','Marca':'marca'},True)

Coluna(s) informada(s) no dicionário foram renomeadas com sucesso!

--- Primeiras 5 Linhas (df.head()) ---


,id,marca
0,1,Apple
1,2,Samsung
2,3,Motorola
3,4,Xiaomi
4,5,LG


In [ ]:
renames_columns_df(produto_df,{'ID_Subcategoria':'id_subcategoria','ID_Produto':'id','Descricao_Produto':'descricao_produto','ID_Marca':'id_marca','Preco_Unitario':'preco_unitario','Tributos':'tributos','Custo':'custo'},True)

Coluna(s) informada(s) no dicionário foram renomeadas com sucesso!

--- Primeiras 5 Linhas (df.head()) ---


,id_subcategoria,id,descricao_produto,id_marca,preco_unitario,tributos,custo
0,10,1010,iPhone 11,1,2899,347.88,550.81
1,10,1011,iPhone 12,1,3499,419.88,699.80
2,10,1012,iPhone 13,1,4099,614.85,655.84
3,10,1013,iPhone 13 Pro,1,4299,558.87,773.82
4,10,1014,iPhone 13 Pro Max,1,4399,615.86,395.91


In [78]:
print("--- Verificando Duplicatas no DataFrame 'produto_df' ---")

# Encontra e exibe as linhas duplicadas na coluna 'id'
duplicatas = produto_df[produto_df.duplicated(subset=['id'], keep=False)]
if not duplicatas.empty:
    print("Duplicatas encontradas na coluna 'id' do DataFrame:")
    print(duplicatas)
    print("\nRemovendo duplicatas (mantendo a primeira ocorrência)...")
    # Remove as duplicatas, mantendo a primeira ocorrência
    produto_df_limpo = produto_df.drop_duplicates(subset=['id'], keep='first')
    print(f"DataFrame original tinha {len(produto_df)} linhas.")
    print(f"DataFrame após remover duplicatas tem {len(produto_df_limpo)} linhas.")
    produto_df = produto_df_limpo # Atualiza o DataFrame para o limpo
else:
    print("Nenhuma duplicata encontrada na coluna 'id' do DataFrame.")

print("-------------------------------------------------------")

# Seu código de inserção no banco de dados viria aqui, usando o 'produto_df' já limpo

--- Verificando Duplicatas no DataFrame 'produto_df' ---
Duplicatas encontradas na coluna 'id' do DataFrame:
     id_subcategoria    id  descricao_produto  id_marca  preco_unitario  \
117               18  1127  Playstation 5 2TB         9            5982   
118               19  1127     Xbox One 512GB        10            2199   

     tributos   custo  
117    478.56  658.02  
118    307.86  351.84  

Removendo duplicatas (mantendo a primeira ocorrência)...
DataFrame original tinha 213 linhas.
DataFrame após remover duplicatas tem 212 linhas.
-------------------------------------------------------


In [ ]:
renames_columns_df(subcategoria_df,{'ID_Categoria':'id_categoria','ID_Subcategoria':'id','Subcategoria':'subcategoria'},True)

Coluna(s) informada(s) no dicionário foram renomeadas com sucesso!

--- Primeiras 5 Linhas (df.head()) ---


,id_categoria,id,subcategoria
0,1,10,IOS
1,1,11,Android
2,2,12,LED
3,2,13,QLED
4,2,14,OLED


In [43]:
renames_columns_df(fato_df,{'Data_Venda':'data_venda','Data_Entrega':'data_entrega','ID_Canal':'id_canal','ID_Cliente':'id_cliente','ID_Pedido':'id_pedido','ID_Produto':'id_produto','Qtde':'qtde','Valor Total':'valor_total'},True)

Coluna(s) informada(s) no dicionário foram renomeadas com sucesso!

--- Primeiras 5 Linhas (df.head()) ---


,data_venda,data_entrega,id_canal,id_cliente,id_pedido,id_produto,qtde,valor_total
0,2018-01-01,2018-01-22,2,5010,66417,1038,15,26445.30
1,2018-01-01,2018-02-05,1,5426,66418,1177,7,34825.91
2,2018-01-01,2018-01-02,2,7499,66419,1198,12,11946.24
3,2018-01-01,2018-02-05,2,8080,66420,1049,25,79176.00
4,2018-01-01,2018-01-28,2,9370,66421,1102,16,95957.60


In [108]:
print("--- Removendo a coluna 'id_pedido' do DataFrame 'fato_df' ---")
if 'id_pedido' in fato_df.columns:
    fato_df = fato_df.drop(columns=['id_pedido'])
    print("Coluna 'id_pedido' removida do DataFrame 'fato_df'.")
else:
    print("A coluna 'id_pedido' não foi encontrada no DataFrame 'fato_df'. Nenhuma ação necessária.")
print("------------------------------------------------------------\n")

--- Removendo a coluna 'id_pedido' do DataFrame 'fato_df' ---
Coluna 'id_pedido' removida do DataFrame 'fato_df'.
------------------------------------------------------------



In [109]:
details_df(fato_df,['head'])


--- Primeiras 5 Linhas (df.head()) ---


,data_venda,data_entrega,id_canal,id_cliente,id_produto,qtde,valor_total
0,2018-01-01,2018-01-22,2,5010,1038,15,26445.30
1,2018-01-01,2018-02-05,1,5426,1177,7,34825.91
2,2018-01-01,2018-01-02,2,7499,1198,12,11946.24
3,2018-01-01,2018-02-05,2,8080,1049,25,79176.00
4,2018-01-01,2018-01-28,2,9370,1102,16,95957.60


### PASSO 4: Carregamento:
#### O Carregamento, irei fazer para o Banco de Dados.

In [23]:
# Configurando banco de dados
db_user = 'postgres'
db_password = '1234'
db_host = 'localhost'
db_database = 'db_financeiro'
db_porta = '5432'


In [24]:
# Criando conexão para o SQLAlchemy
try:
    db_connection_str = f'postgresql://{db_user}:{db_password}@{db_host}:{db_porta}/{db_database}?client_encoding=UTF8'
    db_connection = create_engine(db_connection_str)
except Exception as e:
    print('Erro:',{e})

In [25]:
#conexão com o banco de dados
try:
    connection = psycopg2.connect(host=db_host,database=db_database,user=db_user,password=db_password)
    connection.autocommit = True
    cursor = connection.cursor()
except Exception as e:
    print('Erro:',{e})

In [ ]:
# #Criando schema no banco de dados:
# try:
#     cursor.execute('CREATE SCHEMA DW_FINANCEIRO')
#     # connection.commit()
# except Exception as e:
#     print('Erro:',{e})

In [42]:
details_df(canal_df,['coluna'])


--- Informações de Colunas ---


'Quantidade de Colunas: 2'

"Nomes das Colunas: ['id', 'descricao_canal']"

In [48]:
#Caso necessite:
try:
    cursor.execute('drop table dw_financeiro.canal')
    connection.commit()
except Exception as e:
    print('Erro:',{e})

In [49]:
#Criando Canal no banco de dados:
try:
    cursor.execute('CREATE TABLE IF NOT EXISTS DW_FINANCEIRO.CANAL (id INT PRIMARY KEY,descricao_canal VARCHAR(255));')
    connection.commit()
except Exception as e:
    print('Erro:',{e})

In [50]:
# 3. Usando o to_sql para inserir os dados
try:
    canal_df.to_sql(
        'canal',         # Nome da tabela no banco de dados
        con=db_connection, # Conexão com o banco de dados
        schema='dw_financeiro',   # Esquema (geralmente 'public')
        if_exists='append', # 'append': adiciona novas linhas; 'replace': recria a tabela; 'fail': gera erro se a tabela existir
        index=False        # Não escreve o índice do DataFrame como uma coluna
    )
    print("Dados inseridos com sucesso na tabela canal usando to_sql!")
except Exception as e:
    print(f"Erro ao inserir dados com to_sql: {e}")
finally:
    # Fechando a conexão
    db_connection.dispose()

Dados inseridos com sucesso na tabela canal usando to_sql!


In [40]:
details_df(categoria_df,['coluna'])


--- Informações de Colunas ---


'Quantidade de Colunas: 2'

"Nomes das Colunas: ['id', 'categoria']"

In [53]:
#Caso necessite:
try:
    cursor.execute('drop table dw_financeiro.categoria')
    connection.commit()
except Exception as e:
    print('Erro:',{e})

In [54]:
#Criando Categoria no banco de dados:
try:
    cursor.execute('CREATE TABLE IF NOT EXISTS DW_FINANCEIRO.CATEGORIA (id INT PRIMARY KEY,categoria VARCHAR(255));')
    connection.commit()
except Exception as e:
    print('Erro:',{e})

In [55]:
# 3. Usando o to_sql para inserir os dados
try:
    categoria_df.to_sql(
        'categoria',         # Nome da tabela no banco de dados
        con=db_connection, # Conexão com o banco de dados
        schema='dw_financeiro',   # Esquema (geralmente 'public')
        if_exists='append', # 'append': adiciona novas linhas; 'replace': recria a tabela; 'fail': gera erro se a tabela existir
        index=False        # Não escreve o índice do DataFrame como uma coluna
    )
    print("Dados inseridos com sucesso na tabela categoria usando to_sql!")
except Exception as e:
    print(f"Erro ao inserir dados com to_sql: {e}")
finally:
    # Fechando a conexão
    db_connection.dispose()

Dados inseridos com sucesso na tabela categoria usando to_sql!


In [39]:
details_df(cidade_df,['coluna'])


--- Informações de Colunas ---


'Quantidade de Colunas: 3'

"Nomes das Colunas: ['id', 'cidade', 'uf']"

In [57]:
#Caso necessite:
try:
    cursor.execute('drop table dw_financeiro.cidade')
    connection.commit()
except Exception as e:
    print('Erro:',{e})

In [58]:
#Criando Cidade no banco de dados:
try:
    cursor.execute('CREATE TABLE IF NOT EXISTS DW_FINANCEIRO.CIDADE (id INT PRIMARY KEY,cidade VARCHAR(255),uf VARCHAR(3));')
    connection.commit()
except Exception as e:
    print('Erro:',{e})

In [59]:
# 3. Usando o to_sql para inserir os dados
try:
    cidade_df.to_sql(
        'cidade',         # Nome da tabela no banco de dados
        con=db_connection, # Conexão com o banco de dados
        schema='dw_financeiro',   # Esquema (geralmente 'public')
        if_exists='append', # 'append': adiciona novas linhas; 'replace': recria a tabela; 'fail': gera erro se a tabela existir
        index=False        # Não escreve o índice do DataFrame como uma coluna
    )
    print("Dados inseridos com sucesso na tabela cidade usando to_sql!")
except Exception as e:
    print(f"Erro ao inserir dados com to_sql: {e}")
finally:
    # Fechando a conexão
    db_connection.dispose()

Dados inseridos com sucesso na tabela cidade usando to_sql!


In [38]:
details_df(marca_df,['coluna'])


--- Informações de Colunas ---


'Quantidade de Colunas: 2'

"Nomes das Colunas: ['id', 'marca']"

In [60]:
#Caso necessite:
try:
    cursor.execute('drop table dw_financeiro.marca')
    connection.commit()
except Exception as e:
    print('Erro:',{e})

In [61]:
#Criando Marca no banco de dados:
try:
    cursor.execute('CREATE TABLE IF NOT EXISTS DW_FINANCEIRO.MARCA (id INT PRIMARY KEY,marca VARCHAR(255));')
    connection.commit()
except Exception as e:
    print('Erro:',{e})

In [62]:
# 3. Usando o to_sql para inserir os dados
try:
    marca_df.to_sql(
        'marca',         # Nome da tabela no banco de dados
        con=db_connection, # Conexão com o banco de dados
        schema='dw_financeiro',   # Esquema (geralmente 'public')
        if_exists='append', # 'append': adiciona novas linhas; 'replace': recria a tabela; 'fail': gera erro se a tabela existir
        index=False        # Não escreve o índice do DataFrame como uma coluna
    )
    print("Dados inseridos com sucesso na tabela marca usando to_sql!")
except Exception as e:
    print(f"Erro ao inserir dados com to_sql: {e}")
finally:
    # Fechando a conexão
    db_connection.dispose()

Dados inseridos com sucesso na tabela marca usando to_sql!


In [67]:
details_df(produto_df,['coluna'])


--- Informações de Colunas ---


'Quantidade de Colunas: 7'

"Nomes das Colunas: ['id_subcategoria', 'id', 'descricao_produto', 'id_marca', 'preco_unitario', 'tributos', 'custo']"

In [74]:
#Caso necessite:
try:
    cursor.execute('drop table dw_financeiro.produto')
    connection.commit()
except Exception as e:
    print('Erro:',{e})

In [75]:
#Criando PRODUTO no banco de dados:
try:
    cursor.execute('CREATE TABLE IF NOT EXISTS DW_FINANCEIRO.PRODUTO (id_subcategoria INT NOT NULL,id INT PRIMARY KEY,descricao_produto VARCHAR(255),id_marca INT NOT NULL,preco_unitario NUMERIC(10, 2),tributos NUMERIC(10, 2),custo NUMERIC(10, 2),CONSTRAINT FK_PRODUTO_MARCA FOREIGN KEY (id_marca) REFERENCES DW_FINANCEIRO.MARCA(id),CONSTRAINT FK_PRODUTO_SUBCATEGORIA FOREIGN KEY (id_subcategoria) REFERENCES DW_FINANCEIRO.SUBCATEGORIA(id));')
    connection.commit()
except Exception as e:
    print('Erro:',{e})

In [79]:
# 3. Usando o to_sql para inserir os dados
try:
    produto_df.to_sql(
        'produto',         # Nome da tabela no banco de dados
        con=db_connection, # Conexão com o banco de dados
        schema='dw_financeiro',   # Esquema (geralmente 'public')
        if_exists='append', # 'append': adiciona novas linhas; 'replace': recria a tabela; 'fail': gera erro se a tabela existir
        index=False        # Não escreve o índice do DataFrame como uma coluna
    )
    print("Dados inseridos com sucesso na tabela produto usando to_sql!")
except Exception as e:
    print(f"Erro ao inserir dados com to_sql: {e}")
finally:
    # Fechando a conexão
    db_connection.dispose()

Dados inseridos com sucesso na tabela produto usando to_sql!


In [91]:
details_df(subcategoria_df,['coluna'])


--- Informações de Colunas ---


'Quantidade de Colunas: 3'

"Nomes das Colunas: ['id_categoria', 'id', 'subcategoria']"

In [93]:
#Caso necessite:
try:
    cursor.execute('drop table dw_financeiro.subcategoria cascade')
    connection.commit()
except Exception as e:
    print('Erro:',{e})

In [94]:
#Criando SUBCATEGORIA no banco de dados:
try:
    cursor.execute('CREATE TABLE IF NOT EXISTS DW_FINANCEIRO.SUBCATEGORIA (id_categoria  INT NOT NULL, id INT PRIMARY KEY,subcategoria VARCHAR(255), CONSTRAINT FK_SUBCATEGORIA_CATEGORIA FOREIGN KEY (id_categoria) REFERENCES DW_FINANCEIRO.CATEGORIA(id));')
    connection.commit()
except Exception as e:
    print('Erro:',{e})

In [95]:
# 3. Usando o to_sql para inserir os dados
try:
    subcategoria_df.to_sql(
        'subcategoria',         # Nome da tabela no banco de dados
        con=db_connection, # Conexão com o banco de dados
        schema='dw_financeiro',   # Esquema (geralmente 'public')
        if_exists='append', # 'append': adiciona novas linhas; 'replace': recria a tabela; 'fail': gera erro se a tabela existir
        index=False        # Não escreve o índice do DataFrame como uma coluna
    )
    print("Dados inseridos com sucesso na tabela subcategoria usando to_sql!")
except Exception as e:
    print(f"Erro ao inserir dados com to_sql: {e}")
finally:
    # Fechando a conexão
    db_connection.dispose()

Dados inseridos com sucesso na tabela subcategoria usando to_sql!


In [22]:
details_df(clientes_df,['coluna'])


--- Informações de Colunas ---


'Quantidade de Colunas: 8'

"Nomes das Colunas: ['id', 'nome', 'sobrenome', 'data_nascimento', 'estado_civil', 'genero', 'educacao', 'id_cidade']"

In [46]:
#Caso necessite:
try:
    cursor.execute('drop table dw_financeiro.clientes')
    connection.commit()
except Exception as e:
    print('Erro:',{e})

In [96]:
#Criando CLIENTES no banco de dados:
try:
    cursor.execute('''CREATE TABLE IF NOT EXISTS DW_FINANCEIRO.CLIENTES (
    id INT PRIMARY KEY,
    nome VARCHAR(255) NOT NULL,
    sobrenome VARCHAR(255) NOT NULL,
    data_nascimento DATE NOT NULL,
    estado_civil VARCHAR(50) NOT NULL,
    genero VARCHAR(50) NOT NULL,
    educacao VARCHAR(255) NOT NULL,
    id_cidade INT NOT NULL,
    CONSTRAINT FK_CLIENTES_CIDADE FOREIGN KEY (id_cidade) REFERENCES DW_FINANCEIRO.CIDADE(id)
);''')
    connection.commit()
except Exception as e:
    print('Erro:',{e})

In [105]:
# 3. Usando o to_sql para inserir os dados
try:
    clientes_df.to_sql(
        'clientes',         # Nome da tabela no banco de dados
        con=db_connection, # Conexão com o banco de dados
        schema='dw_financeiro',   # Esquema (geralmente 'public')
        if_exists='append', # 'append': adiciona novas linhas; 'replace': recria a tabela; 'fail': gera erro se a tabela existir
        index=False        # Não escreve o índice do DataFrame como uma coluna
    )
    print("Dados inseridos com sucesso na tabela clientes usando to_sql!")
except Exception as e:
    print(f"Erro ao inserir dados com to_sql: {e}")
finally:
    # Fechando a conexão
    db_connection.dispose()

Dados inseridos com sucesso na tabela clientes usando to_sql!


In [110]:
details_df(fato_df,['coluna'])


--- Informações de Colunas ---


'Quantidade de Colunas: 7'

"Nomes das Colunas: ['data_venda', 'data_entrega', 'id_canal', 'id_cliente', 'id_produto', 'qtde', 'valor_total']"

In [111]:
#Criando schema no banco de dados:
try:
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS DW_FINANCEIRO.FATO (
            id SERIAL PRIMARY KEY,
            data_venda DATE NOT NULL,
            data_entrega DATE NOT NULL,
            id_canal INT NOT NULL,
            id_cliente INT NOT NULL,
            id_produto INT NOT NULL,
            qtde INT NOT NULL,
            valor_total NUMERIC(15, 2),
            CONSTRAINT FK_FATO_CANAL FOREIGN KEY (id_canal) REFERENCES DW_FINANCEIRO.CANAL(id),
            CONSTRAINT FK_FATO_CLIENTE FOREIGN KEY (id_cliente) REFERENCES DW_FINANCEIRO.CLIENTES(id),
            CONSTRAINT FK_FATO_PRODUTO FOREIGN KEY (id_produto) REFERENCES DW_FINANCEIRO.PRODUTO(id)
        );
    ''')
    connection.commit()
except Exception as e:
    print('Erro:',{e})

In [112]:
# 3. Usando o to_sql para inserir os dados
try:
    fato_df.to_sql(
        'fato',         # Nome da tabela no banco de dados
        con=db_connection, # Conexão com o banco de dados
        schema='dw_financeiro',   # Esquema (geralmente 'public')
        if_exists='append', # 'append': adiciona novas linhas; 'replace': recria a tabela; 'fail': gera erro se a tabela existir
        index=False        # Não escreve o índice do DataFrame como uma coluna
    )
    print("Dados inseridos com sucesso na tabela clientes usando to_sql!")
except Exception as e:
    print(f"Erro ao inserir dados com to_sql: {e}")
finally:
    # Fechando a conexão
    db_connection.dispose()

Dados inseridos com sucesso na tabela clientes usando to_sql!
